In [0]:
%pip install \
    azure-keyvault-secrets==4.7.0 \
    azure-identity==1.15.0 \
    azure-core==1.29.5 \
    azure-storage-file-datalake==12.14.0 \
    sseclient-py \
    openai \
    dotenv \
    confluent-kafka

In [0]:
dbutils.library.restartPython()

In [0]:
import sys
from pathlib import Path

# vault_manager.py 경로 추가
sys.path.append(str(Path(__file__).resolve().parents[1] / "src" / "utils"))

from vault_manager import vault

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC ## PULSE — Wikipedia EventStreams 수신 + AI 규칙 생성
# MAGIC - Wikipedia SSE → Bronze 적재
# MAGIC - AI 규칙 생성 (최초 1회)
# MAGIC - GX 품질 검사

# COMMAND ----------
import os
import sys
import json
import requests
from sseclient import SSEClient
from datetime import datetime

# ── 환경 설정 ────────────────────────────────────────────
os.environ["KEY_VAULT_URL"] = "https://kv-sense-team4.vault.azure.net/"
sys.path.insert(0, "/Workspace/Repos/3dt030@msacademy.msai.kr/3dt-3nd-project/src")

# ── vault 초기화 ─────────────────────────────────────────
import utils.vault_manager
utils.vault_manager._instance = None
from utils.vault_manager import get_vault_manager

vault = get_vault_manager()

# ── 연결 및 시크릿 로드 ──────────────────────────────────
storage_client    = vault.get_storage_client("datacopsadls")
kafka_producer    = vault.get_kafka_producer()

gx_openai_key        = vault.get_secret("gx-rulegen-openai-key")
gx_openai_endpoint   = vault.get_secret("gx-rulegen-openai-endpoint")
gx_openai_deployment = vault.get_secret("gx-rulegen-deployment-gpt-4-1-mini")
gx_openai_api_version = "2025-01-01-preview"

# ── 확인 ─────────────────────────────────────────────────
print("[OK] Key Vault 연결 완료")
print("[OK] ADLS 연결 완료")
print("[OK] Kafka Producer 연결 완료")
print("[OK] GX RuleGen OpenAI 설정 로드 완료")
print(f"[INFO] deployment = {gx_openai_deployment}")

In [0]:
import json
import requests
import time
from sseclient import SSEClient

TOPIC = "wiki-raw-events"
WIKI_STREAM_URL = "https://stream.wikimedia.org/v2/stream/recentchange"
HEADERS = {
    "Accept": "text/event-stream",
    "User-Agent": "datacops-data-quality/0.1"
}

def delivery_report(err, msg):
    if err:
        print(f"[FAIL] 전송 실패: {err}")
    else:
        print(f"[OK] offset={msg.offset()} | partition={msg.partition()}")

def collect_and_produce(stream_url, max_count=1000, filters=None, max_retries=3):
    events = []
    count = 0
    retry = 0

    while count < max_count and retry < max_retries:
        try:
            response = requests.get(stream_url, stream=True, headers=HEADERS, timeout=30)
            client = SSEClient(response)

            for event in client.events():
                if event.event != "message":
                    continue
                try:
                    data = json.loads(event.data)
                except json.JSONDecodeError:
                    continue

                if filters:
                    if not all(data.get(k) == v for k, v in filters.items()):
                        continue

                events.append(data)
                kafka_producer.produce(
                    topic=TOPIC,
                    key=str(data.get("id", "")),
                    value=json.dumps(data),
                    callback=delivery_report
                )
                kafka_producer.poll(0)
                count += 1

                if count >= max_count:
                    break

        except Exception as e:
            retry += 1
            print(f"[WARN] 연결 끊김 ({retry}/{max_retries}): {e}")
            time.sleep(2)
            continue

    kafka_producer.flush()
    print(f"[완료] {count}건 Kafka({TOPIC}) 전송 완료")
    return events

print("Wikipedia 이벤트 수집 시작...")
raw_events = collect_and_produce(stream_url=WIKI_STREAM_URL, max_count=1000)
print(f"[OK] raw_events {len(raw_events)}건 저장 완료")

### 카프카를 통해 가져온 데이터 확인

In [0]:
from confluent_kafka import Consumer
import json

consumer = Consumer({
    "bootstrap.servers": vault.get_secret("kafka-bootstrap-servers"),
    "security.protocol": "SASL_PLAINTEXT",
    "sasl.mechanism":    "SCRAM-SHA-256",
    "sasl.username":     vault.get_secret("kafka-username"),
    "sasl.password":     vault.get_secret("kafka-password"),
    "group.id":          "check-fresh",
    "auto.offset.reset": "earliest",
})

consumer.subscribe(["wiki-raw-events"])

count = 0
while count < 3:
    msg = consumer.poll(timeout=5.0)
    if msg is None:
        continue
    if msg.error():
        print(f"[ERROR] {msg.error()}")
        break
    data = json.loads(msg.value())
    print(f"\n{'='*50}")
    print(f"title   : {data.get('title')}")
    print(f"wiki    : {data.get('wiki')}")
    print(f"type    : {data.get('type')}")
    print(f"user    : {data.get('user')}")
    print(f"bot     : {data.get('bot')}")
    print(f"comment : {data.get('comment')}")
    print(f"length  : {data.get('length')}")
    print(f"revision: {data.get('revision')}")
    count += 1

consumer.close()
print(f"\n[완료] {count}건 확인")

### 가져온 데이터를 통해 AI 규칙 생성

In [0]:
# COMMAND ----------
import pandas as pd

def detect_column_type(series: pd.Series) -> str:
    """
    컬럼 타입 자동 감지
    """
    non_null = series.dropna()

    if non_null.empty:
        return "unknown"

    if pd.api.types.is_bool_dtype(non_null):
        return "boolean"

    if pd.api.types.is_numeric_dtype(non_null):
        return "numeric"

    # 문자열 기반 timestamp 감지
    sample = non_null.astype(str).head(20)
    parsed = pd.to_datetime(sample, errors="coerce", utc=True)
    if parsed.notna().mean() >= 0.8:
        return "timestamp"

    unique_ratio = non_null.nunique() / len(non_null)

    if unique_ratio < 0.05:
        return "categorical"

    return "string"


def safe_sample_values(series: pd.Series, n=5):
    """
    JSON 직렬화 가능한 샘플 값 반환
    """
    values = series.dropna().head(n).tolist()
    result = []

    for v in values:
        try:
            json.dumps(v)
            result.append(v)
        except TypeError:
            result.append(str(v))

    return result


def auto_profile(data: list[dict]) -> dict:
    """
    어떤 데이터든 받아서 컬럼 정보 자동 분석
    범용 설계 — 데이터 구조에 의존하지 않음
    """
    nullable_cols = [
        col for col, info in profile.items()
        if 0.05 < info["null_rate"] < 0.95 and col in df.columns
    ]
    categorical_cols = [
    col for col, info in profile.items()
    if info["dtype"] in ["categorical", "boolean"]
    and info["null_rate"] < 0.05
    and col in df.columns
    and 1 < info["unique_count"] < 20  # 단일값/고카디널리티 제외
    ]

    profile = {}

    for col in df.columns:
        series = df[col]
        non_null = series.dropna()
        dtype = detect_column_type(series)

        if correlations:
            profile[null_col]["null_when"] = correlations
            print(f"  [상관관계 발견] {null_col}: {correlations}")

    return profile

def auto_profile(data: list[dict]) -> dict:
    df = pd.json_normalize(data)
    profile = {}
    for col in df.columns:
        series = df[col]
        non_null = series.dropna()
        dtype = detect_column_type(series)
        col_info = {
            "dtype": dtype,
            "null_rate": round(series.isna().mean(), 3),
            "unique_count": int(non_null.nunique()),
            "sample": safe_sample_values(series, n=5)
        }

        if dtype == "numeric" and not non_null.empty:
            col_info.update({
                "min": float(non_null.min()),
                "max": float(non_null.max()),
                "mean": round(float(non_null.mean()), 3)
            })

        if dtype in ["string", "categorical"] and not non_null.empty:
            lengths = non_null.astype(str).str.len()
            col_info.update({
                "min_length": int(lengths.min()),
                "max_length": int(lengths.max()),
                "avg_length": round(float(lengths.mean()), 2)
            })
        profile[col] = col_info

    # 상관관계 계산 추가
    print("[INFO] NULL 상관관계 분석 중...")
    profile = compute_null_correlations(df, profile)

    return profile


profile = auto_profile(raw_events)

print(f"[OK] 컬럼 {len(profile)}개 분석 완료")

for col, info in list(profile.items())[:10]:
    print(f"  {col}: {info['dtype']} | null={info['null_rate']} | unique={info['unique_count']}")

In [0]:
# COMMAND ----------
# null_rate 0.95 이상 컬럼만 제외하고 전체 전달

slim_profile = {
    col: info
    for col, info in profile.items()
    if info["null_rate"] < 0.95
}

print(f"[INFO] 전체 {len(profile)}개 → AI 전달 {len(slim_profile)}개 컬럼")
print(f"[INFO] 제외된 컬럼 ({len(profile) - len(slim_profile)}개):")
for col, info in profile.items():
    if info["null_rate"] >= 0.95:
        print(f"  - {col}: null={info['null_rate']}")

print("\n[INFO] 비즈니스 의미 NULL 발견된 컬럼:")
for col, info in slim_profile.items():
    if "null_when" in info:
        print(f"  - {col}: {info['null_when']}")

# ── 도메인 자동 감지 ──────────────────────────────────────
from openai import AzureOpenAI

_client = AzureOpenAI(
    api_key=gx_openai_key,
    azure_endpoint=gx_openai_endpoint,
    api_version=gx_openai_api_version,
)

def detect_domain(profile: dict) -> dict:
    """
    프로파일 컬럼명 + 샘플값으로 도메인 자동 감지
    규칙 생성 전에 실행해서 도메인 컨텍스트 확보
    어떤 도메인 데이터든 자동 감지 가능
    """
    compact = json.dumps(
        {
            col: {
                "dtype": info["dtype"],
                "sample": info["sample"]
            }
            for col, info in list(profile.items())[:15]
        },
        ensure_ascii=False,
        indent=2
    )

    response = _client.chat.completions.create(
        model=gx_openai_deployment,
        messages=[
            {
                "role": "system",
                "content": """You are a data domain expert.
Identify the data domain from column names, dtypes, and sample values.
Return a JSON with:
- domain_name: short snake_case name (e.g. wikipedia_recentchange, nyc_taxi_trips, health_checkup)
- domain_description: one sentence describing the data
- key_columns: list of 3-5 most important columns
- data_characteristics: list of 2-3 notable characteristics

Return ONLY valid JSON. No markdown."""
            },
            {
                "role": "user",
                "content": f"Column profile:\n{compact}\n\nIdentify the domain."
            }
        ],
        temperature=0,
        max_tokens=200,
    )

    raw = response.choices[0].message.content.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {
            "domain_name": "unknown_domain",
            "domain_description": "Unknown domain",
            "key_columns": [],
            "data_characteristics": []
        }

# 실행
domain_info = detect_domain(slim_profile)
domain_name = domain_info["domain_name"]

print(f"\n[OK] 도메인 감지 완료")
print(f"  domain     : {domain_name}")
print(f"  description: {domain_info['domain_description']}")
print(f"  key_columns: {domain_info['key_columns']}")
print(f"  특성       : {domain_info['data_characteristics']}")

### AI 규칙 생성 프롬프트 작성

In [0]:
# COMMAND ----------

def build_stage1_prompt(profile: dict, domain_name: str) -> list[dict]:
    """
    1단계: 전체 컬럼 → NULL 전략 + 텍스트 품질 컬럼 식별
    null_when 필드로 도메인 무관하게 비즈니스 의미 NULL 자동 판단
    """
    compact_profile = json.dumps(profile, ensure_ascii=False, indent=2)

    system_prompt = """
You are a senior data quality engineer building a domain-agnostic automated data quality platform.
The platform works for ANY domain by inferring context from column names, dtypes, null rates, and sample values.

== NULL HANDLING STRATEGY ==
- drop             : critical identifier (name contains 'id', 'uuid', 'key', 'request_id') — drop row if null
- allow            : null has business meaning
- fill_default     : fill with fixed value (specify default_value)
- fill_mean        : fill with mean (normally distributed numeric)
- fill_median      : fill with median (skewed numeric)
- fill_mode        : fill with most frequent value (categorical/boolean)
- fill_forward     : fill with previous row value (time-ordered)
- fill_backward    : fill with next row value
- fill_interpolate : linear interpolation (ordered numeric)
- fill_conditional : fill based on another column value

== NULL CORRELATION RULE (HIGHEST PRIORITY) ==
If a column has "null_when" field:
- This means NULL is strongly correlated with a specific value in another column
- This NULL has BUSINESS MEANING → ALWAYS use "allow", never fill
- Example: null_when: {"type=new": 0.98} means NULL when type=new → new page creation → allow
- This rule overrides ALL other rules

== OTHER NULL RULES ==
- Identifier columns (name contains 'id', 'uuid', 'key', 'request_id'): ALWAYS "drop"
- Boolean columns (minor, patrolled, bot): "fill_mode"
- null_rate > 0.8 columns (log_*, subtypes): "allow"
- Free-text columns (comment, title, description, parsedcomment): "fill_default" with ""
- Timestamp columns: "fill_forward" or "drop", NEVER "fill_mean" or "fill_median"
- Categorical columns with null_rate > 0.1: "fill_mode"
- Skewed numeric (high std relative to mean): "fill_median"
- Normal numeric: "fill_mean"

Return ONLY valid JSON. No markdown, no explanation, no comments.
"""

    user_prompt = f"""
Domain: {domain_name}

Column profile (dtype, null_rate, unique_count, sample, mean/std for numeric, null_when for correlated nulls):
{compact_profile}

Return:
{{
  "null_strategies": {{
    "<col>": {{
      "strategy": "drop|allow|fill_default|fill_mean|fill_median|fill_mode|fill_forward|fill_backward|fill_interpolate|fill_conditional",
      "default_value": null,
      "condition_column": null,
      "condition_map": null,
      "reason": "..."
    }}
  }},
  "text_quality_columns": [
    {{
      "column": "...",
      "checks": ["profanity", "spam", "hate_speech", "pii"],
      "reason": "..."
    }}
  ]
}}
"""
    return [
        {"role": "system", "content": system_prompt.strip()},
        {"role": "user", "content": user_prompt.strip()}
    ]


def build_stage2_prompt(profile: dict, domain_name: str) -> list[dict]:
    """
    2단계: 핵심 컬럼만 → 검증 규칙 + 이상치 탐지
    null_rate < 0.1 또는 categorical/boolean/timestamp 위주
    """
    key_profile = {
        col: info for col, info in profile.items()
        if info["null_rate"] < 0.1
        or info["dtype"] in ["categorical", "boolean", "timestamp"]
    }
    compact_profile = json.dumps(key_profile, ensure_ascii=False, indent=2)

    system_prompt = """
You are a senior data quality engineer.
Generate stable validation rules and anomaly detection rules.

== VALIDATION RULES ==
- Do NOT generate min/max range rules (sample-dependent)
- Do NOT generate string length rules (sample-dependent)
- DO generate: not null checks, type checks, allowed value sets (cardinality < 10), datetime format
- severity: critical / warning / info
- One rule per expectation type per column, no duplicates
- Categorical with high unique_count (>10) or evolving over time: WARNING not CRITICAL
- For value_set rules on columns with unique_count > 3: use WARNING not CRITICAL
- If a column has "null_when" field in profile: do NOT generate expect_column_values_to_not_be_null for it

== ANOMALY RULES ==
- delta    : size/length change columns → flag extreme changes
- zscore   : normal numeric (use mean/std from profile) → flag beyond 3 std
- iqr      : skewed numeric → flag outside 1.5*IQR
- frequency: user/bot activity → flag abnormal rates
- Always specify numeric threshold where possible

Return ONLY valid JSON. No markdown, no explanation, no comments.
"""

    user_prompt = f"""
Domain: {domain_name}
Profile:
{compact_profile}

Return:
{{
  "suite_name": "{domain_name}_quality_suite",
  "domain": "{domain_name}",
  "generated_at": "2024-01-01T00:00:00Z",
  "expectations": [
    {{
      "expectation_type": "...",
      "column": "...",
      "kwargs": {{}},
      "severity": "critical|warning|info",
      "reason": "..."
    }}
  ],
  "anomaly_rules": [
    {{
      "name": "...",
      "columns": ["..."],
      "method": "zscore|iqr|delta|frequency",
      "threshold": null,
      "severity": "critical|warning",
      "reason": "..."
    }}
  ]
}}
"""
    return [
        {"role": "system", "content": system_prompt.strip()},
        {"role": "user", "content": user_prompt.strip()}
    ]

# Redis 키 도메인 기반으로 자동 설정
CACHE_KEY_RULES  = f"gx_rules:{domain_name}"
CACHE_KEY_SCHEMA = f"gx_schema:{domain_name}"

# 프롬프트 생성
messages_stage1 = build_stage1_prompt(slim_profile, domain_name)
messages_stage2 = build_stage2_prompt(slim_profile, domain_name)
print(f"[OK] 1단계 프롬프트 준비 완료 (도메인: {domain_name})")
print(f"[OK] 2단계 프롬프트 준비 완료 (도메인: {domain_name})")
print(f"[INFO] 캐시 키: {CACHE_KEY_RULES}")

In [0]:
# COMMAND ----------
def build_gx_rule_prompt(profile: dict, domain_name: str = "wikipedia_recentchange") -> list[dict]:
    """
    컬럼 프로파일을 기반으로 GX 규칙 JSON 생성을 요청하는 프롬프트 생성
    """
    compact_profile = json.dumps(profile, ensure_ascii=False, indent=2)

    system_prompt = """
You are a senior data quality engineer.
Your task is to generate data quality validation rules for Great Expectations.

Return ONLY valid JSON.
Do not include markdown.
Do not include explanations.
Do not include comments.
"""

    user_prompt = f"""
We are building an AI-based data quality rule generation pipeline.

Data domain:
{domain_name}

Input data:
Wikipedia EventStreams recentchange events.

Column profile:
{compact_profile}

Generate a Great Expectations-like validation rule JSON.

Requirements:
1. Return only valid JSON.
2. The top-level JSON must include:
   - suite_name
   - domain
   - generated_at
   - expectations
3. Each expectation must include:
   - expectation_type
   - column
   - kwargs
   - severity
   - reason
4. Use only columns that exist in the provided profile.
5. Avoid overfitting to sample values.
6. Generate practical rules for:
   - not null checks
   - type checks
   - value range checks for numeric columns
   - allowed values for low-cardinality categorical columns
   - string length checks
   - timestamp validity checks
7. Use severity values among:
   - critical
   - warning
   - info
8. Do not generate rules for columns with very high null rate unless the rule is only a warning.
"""

    return [
        {"role": "system", "content": system_prompt.strip()},
        {"role": "user", "content": user_prompt.strip()}
    ]


messages = build_gx_rule_prompt(profile)
print("[OK] GX 규칙 생성 프롬프트 준비 완료")